# 🚀 Columnar Storage (Parquet), DuckDB SQL & Point-in-Time Joins

In this notebook, we explore:
1. **CSV vs. Compressed Parquet Benchmarks** (disk footprint and read latency).
2. **DuckDB SQL Window Functions** (rolling moving averages without loading entire files into memory).
3. **Point-in-Time (ASOF) Feature Merges** to prevent temporal data leakage.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl

src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from duckdb_parquet import export_to_parquet
from temporal_joins import point_in_time_join

print("Modules imported successfully!")

### 1. Benchmark: CSV vs Parquet File Footprint

In [ ]:
n_rows = 100_000
demo_df = pd.DataFrame({
    "device_id": np.random.choice(["sensor_A", "sensor_B", "sensor_C"], size=n_rows),
    "timestamp": pd.date_range("2024-01-01", periods=n_rows, freq="s"),
    "temperature": np.random.normal(25.0, 5.0, size=n_rows),
    "pressure": np.random.normal(1013.25, 20.0, size=n_rows)
})

csv_path = Path("benchmark_sample.csv")
parquet_path = Path("benchmark_sample.parquet")

demo_df.to_csv(csv_path, index=False)
export_to_parquet(demo_df, parquet_path)

csv_size_kb = csv_path.stat().st_size / 1024.0
parquet_size_kb = parquet_path.stat().st_size / 1024.0

print(f"CSV Size:     {csv_size_kb:.1f} KB")
print(f"Parquet Size: {parquet_size_kb:.1f} KB (Snappy Compressed)")
print(f"Storage Reduction: {((csv_size_kb - parquet_size_kb) / csv_size_kb) * 100:.1f}%")

# Clean up temporary benchmark files
csv_path.unlink(missing_ok=True)
parquet_path.unlink(missing_ok=True)

### 2. Point-in-Time Correct (ASOF) Join Demonstration

In [ ]:
from datetime import datetime

events = pl.DataFrame({
    "device_id": ["sensor_1"],
    "timestamp": [datetime(2024, 1, 1, 12, 30, 0)],
    "target_failure": [1]
})

feature_history = pl.DataFrame({
    "device_id": ["sensor_1", "sensor_1", "sensor_1"],
    "timestamp": [
        datetime(2024, 1, 1, 12, 0, 0),
        datetime(2024, 1, 1, 12, 25, 0),  # Valid match (latest before 12:30)
        datetime(2024, 1, 1, 12, 35, 0)   # Future state (must NOT leak)
    ],
    "vibration_level": [1.2, 4.8, 12.5]
})

pit_result = point_in_time_join(events, feature_history)
print("Point-in-Time Join Result (Zero Future Leakage):")
print(pit_result)